<a href="https://colab.research.google.com/github/manurudeepika02-del/Infosys_FreightQuote_AI/blob/main/FreightQuote_AI_Milestone3_Combined_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ FreightQuote AI — Combined Milestone 1 & 2 Notebook
### Integration Notebook (Milestone 3 Requirement)

This notebook merges:
- **Part A — Milestone 1:** Login Page (Streamlit + JWT + bcrypt + OTP email auth)
- **Part B — Milestone 2:** Enterprise Multi-Agent Logistics Intelligence Platform


---
## 🅰️ PART A — Milestone 1: Login Page
---

In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.9 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, random, smtplib
from email.mime.text import MIMEText
import streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# ============================================================
# EMAIL CONFIG (fill these in, or set as Colab secrets and read via userdata)
# ============================================================
def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

SMTP_EMAIL = _get_secret("EMAIL_ID") or "noreply.infosysspringboard@gmail.com"
SMTP_APP_PASSWORD = _get_secret("EMAIL_PASSWORD")  # matches your existing Colab Secrets

def send_otp_email(to_email, otp):
    try:
        msg = MIMEText(f"Your Infosys Freight Quote Portal verification code is: {otp}\n\nThis code expires in 5 minutes.")
        msg["Subject"] = "Infosys Freight Quote Portal - Password Reset OTP"
        msg["From"] = SMTP_EMAIL
        msg["To"] = to_email
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.sendmail(SMTP_EMAIL, to_email, msg.as_string())
        return True
    except Exception as e:
        st.error(f"Email send failed: {e}")
        return False

# ============================================================
# PAGE / THEME CONFIG — Classic Navy & Gold Professional Theme
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#c9a24b"\nbackgroundColor="#f4f5f7"\nsecondaryBackgroundColor="#ffffff"\ntextColor="#1b2436"\n')

st.set_page_config(page_title="Infosys Freight Quote Portal", page_icon="🏛️", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "navy_deep":   "#0b1530",
    "navy":        "#0f1c3f",
    "navy_light":  "#1c2e5c",
    "gold":        "#c9a24b",
    "gold_hover":  "#b8912f",
    "gold_light":  "#e8d9ad",
    "bg_main":     "#f4f5f7",
    "bg_card":     "#ffffff",
    "text_main":   "#1b2436",
    "text_heading":"#0b1530",
    "text_muted":  "#5b6478",
    "text_on_navy":"#f4f5f7",
    "border":      "#d8dbe2",
    "success":     "#2f8f5b",
    "danger":      "#b3413a",
}

JWT_SECRET = _get_secret("JWT_SECRET") or "dev-only-fallback-not-for-production"

# ============================================================
# CLASSIC PROFESSIONAL CSS
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700;800&family=Inter:wght@300;400;500;600;700&display=swap');

    html, body, .stApp {{
        background: {COLORS['bg_main']} !important;
        font-family: 'Inter', sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}

    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['navy']} !important; border: 1px solid {COLORS['gold']} !important;
        border-radius: 6px !important; padding: 6px !important; margin: 8px !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['gold']} !important; color: {COLORS['gold']} !important; stroke: {COLORS['gold']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}

    h1, h2, h3, h4 {{
        font-family: 'Playfair Display', serif !important;
        color: {COLORS['text_heading']} !important;
        letter-spacing: 0.3px;
    }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; font-size: 13px !important; letter-spacing: 0.2px; }}

    /* Inputs — classic bordered style, forced light background everywhere */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: {COLORS['bg_card']} !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"], div[data-baseweb="popover"], div[data-baseweb="menu"], ul[data-baseweb="menu"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1.5px solid {COLORS['border']} !important;
        border-radius: 6px !important;
    }}
    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['gold']} !important;
        box-shadow: 0 0 0 3px rgba(201,162,75,0.18) !important;
    }}
    input, textarea, div[data-baseweb="select"] span, li[role="option"] {{
        color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important;
        background-color: {COLORS['bg_card']} !important;
    }}
    li[role="option"]:hover {{ background-color: {COLORS['gold_light']} !important; }}

    /* Buttons — deep navy with gold accent border, classic feel */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['navy']} !important; color: {COLORS['gold_light']} !important;
        border: 1px solid {COLORS['navy']} !important; border-radius: 6px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 600 !important; font-size: 14px !important;
        height: 46px !important; min-height: 46px !important; letter-spacing: 0.4px;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['gold']} !important; color: {COLORS['navy_deep']} !important;
        border-color: {COLORS['gold']} !important;
    }}

    section[data-testid="stSidebar"] {{
        background: linear-gradient(180deg, {COLORS['navy']} 0%, {COLORS['navy_deep']} 100%) !important;
        border-right: 1px solid {COLORS['gold']} !important;
    }}
    section[data-testid="stSidebar"] * {{ color: {COLORS['text_on_navy']} !important; }}

    .pn-card {{
        background: {COLORS['bg_card']};
        border: 1px solid {COLORS['border']};
        border-top: 3px solid {COLORS['gold']};
        border-radius: 10px;
        padding: 24px;
        box-shadow: 0 2px 10px rgba(11,21,48,0.06);
    }}

    .divider-gold {{ height: 2px; background: linear-gradient(90deg, transparent, {COLORS['gold']}, transparent); margin: 12px 0 20px; }}

    h1.freight-banner-title {{ color: #ffffff !important; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# DB
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None),
             ("otp_code", None), ("otp_expiry", None), ("otp_sent_to", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Enterprise Intelligence Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.2rem 0 0.6rem;">
        <div style="font-size:34px;margin-bottom:6px;">🏛️</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Freight Quote Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:13px;margin:4px 0 0;letter-spacing:0.5px;">{sub}</p>
    </div>
    <div class="divider-gold" style="max-width:120px;margin-left:auto;margin-right:auto;"></div>
    <div style="text-align:center;margin-bottom:1.2rem;"><span style="font-size:1.05rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Freight Quote Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname.strip():
                    st.error("⚠️ Username is required.")
                elif not email.strip():
                    st.error("⚠️ Email is required.")
                elif len(pwd) < 8:
                    st.error(f"⚠️ Password must be at least 8 characters (yours is {len(pwd)}).")
                elif not sa.strip():
                    st.error("⚠️ Security answer is required.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            # ---- Step 0: choose method ----
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT id FROM users WHERE email=?", (email,)).fetchone()
                    if not r:
                        st.error("❌ Email not found.")
                    else:
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(email, otp):
                            st.session_state.otp_code = otp
                            st.session_state.otp_expiry = time.time() + 300  # 5 min
                            st.session_state.otp_sent_to = email
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.success(f"✅ OTP sent to {email}")
                            time.sleep(1)
                            st.rerun()

            # ---- Step 1: security question flow ----
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                # ---- Step 1b: OTP flow ----
                elif st.session_state.get("reset_mode") == "otp":
                    st.info(f"📧 A 6-digit code was sent to **{st.session_state.otp_sent_to}**. It expires in 5 minutes.")
                    entered_otp = st.text_input("Enter OTP", max_chars=6, placeholder="6-digit code")
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    col_verify, col_resend = st.columns(2)
                    if col_verify.button("Verify & Reset →", use_container_width=True):
                        if time.time() > (st.session_state.otp_expiry or 0):
                            st.error("❌ OTP expired. Please request a new one.")
                        elif entered_otp != st.session_state.otp_code:
                            st.error("❌ Incorrect OTP.")
                        elif len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                            st.success("✅ Password updated successfully!")
                            time.sleep(1)
                            st.session_state.reset_email = None
                            st.session_state.otp_code = None
                            navigate("Login")

                    if col_resend.button("Resend OTP", use_container_width=True):
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(st.session_state.reset_email, otp):
                            st.session_state.otp_code = otp
                            st.session_state.otp_expiry = time.time() + 300
                            st.success("✅ New OTP sent.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_code = None
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:20px 8px;text-align:center;">
            <div style="font-size:26px;">🏛️</div>
            <div style="font-weight:700;font-size:16px;font-family:'Playfair Display',serif;color:#ffffff;">Infosys Freight Quote Portal</div>
            <div style="font-size:11px;color:#b7bfd6;letter-spacing:0.5px;">{"ADMIN PANEL" if email=="infosys@ai" else "ENTERPRISE ANALYTICS"}</div>
        </div><hr style="border-color:{COLORS['gold']};opacity:0.3;">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={
                               "container": {"background-color": "transparent"},
                               "icon": {"color": COLORS['gold']},
                               "nav-link": {"color": "#d7dcea", "font-family": "Inter"},
                               "nav-link-selected": {"background-color": COLORS['gold'], "color": COLORS['navy_deep']}
                           })
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    header_bg = f"linear-gradient(90deg, {COLORS['navy_deep']} 0%, {COLORS['navy']} 100%)"

    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{header_bg};border-radius:12px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;border-bottom:3px solid {COLORS['gold']};">
            <div><h1 class="freight-banner-title" style="margin:0;font-size:24px !important;">🏛️ Infosys Freight Quote Portal</h1><div style="color:#b7bfd6;font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['gold']};padding:8px 18px;border-radius:6px;font-weight:700;color:{COLORS['navy_deep']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:32px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

    else:
        st.markdown(f"""
        <div style="background:{header_bg};border-radius:12px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;border-bottom:3px solid {COLORS['gold']};">
            <div><h1 class="freight-banner-title" style="margin:0;font-size:24px !important;">🏛️ Infosys Freight Quote Portal</h1><div style="color:#b7bfd6;font-size:13px;">Enterprise Analytics Dashboard</div></div>
            <div style="background:{COLORS['gold']};padding:8px 18px;border-radius:6px;font-weight:700;color:{COLORS['navy_deep']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:26px;">{icon}</div>
                <div style="font-size:24px;font-weight:700;color:{COLORS['text_heading']};font-family:'Playfair Display',serif;">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;letter-spacing:0.3px;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['gold']}, "bgcolor": COLORS['bg_card'], "borderwidth": 1, "bordercolor": COLORS['navy']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

Writing app.py


In [3]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://snitch-disperser-sterling.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.



🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑
Received Ctrl+C / Stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.


---
## 🅱️ PART B — Milestone 2: Multi-Agent Logistics Platform
---

# ⚡ FreightQuote AI — Milestone 2
### Enterprise Multi-Agent Logistics Intelligence Platform


## Step 1 — Install Dependencies


In [4]:
!pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.9 MB/s eta 0:00:00


## Step 2 — Configure Secrets & Mount Google Drive


In [5]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "freightquote_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FreightQuote_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
except Exception as e:
    STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")

os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
print(f"📁 Storage: {STORAGE_DIR}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")


Mounted at /content/drive
✅ Google Drive mounted.
📁 Storage: /content/drive/MyDrive/FreightQuote_AI
🔑 HF_TOKEN: ✅
🔑 ngrok:    ✅


## Step 3 — Verify GPU & Load Qwen-2.5-3B (4-bit NF4)


In [6]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all 7 secrets (set these in Colab Secrets panel) ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "freightquote_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose Kaggle credentials for the kaggle library
if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# ── Mount Google Drive (auto-detected in Colab) ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FreightQuote_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FreightQuote_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_KEY else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.

📁 Storage:  /content/drive/MyDrive/FreightQuote_AI
🔑 JWT:      ✅ from Colab Secrets
🔑 Admin:    infosys@ai
🔑 HF_TOKEN: ✅
🔑 Kaggle:   ✅
🔑 ngrok:    ✅
🔑 Email:    ✅


In [7]:
!nvidia-smi


Mon Aug  3 09:17:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen-2.5-3B loaded. Footprint (GB): 2.01


## Step 4 — Write All Application Modules (`llm_engine`, `config`, `auth`, `db`, `agents`, `dashboard`)


In [9]:
%%writefile llm_engine.py
"""
llm_engine.py — FreightQuote AI (v4 FINAL — Maximum Speed Edition)
Qwen-2.5-3B-Instruct (4-bit NF4) with:
  • Google Drive Persistent Caching (hf_cache) — instant reload without re-download
  • low_cpu_mem_usage=True + attn_implementation="sdpa" (falls back to "eager") — faster load AND faster generation on T4
  • torch.inference_mode() + use_cache=True + greedy decode — ~1 sec responses
  • Single-Pass generate_debate_and_synthesis() — all 3 agents + synthesis in ~1.5 sec
  • Trimmed max_new_tokens across all 3 generation functions for lower per-call latency
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FreightQuote_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:          # someone else finished loading while we waited
            return _model, _tokenizer
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kw = {"token": HF_TOKEN, "cache_dir": CACHE_DIR} if HF_TOKEN else {"cache_dir": CACHE_DIR}
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
        # sdpa (PyTorch's built-in scaled-dot-product-attention kernel) generates
        # noticeably faster than "eager" on T4 -- eager only wins on load time.
        # Fall back to eager automatically if this transformers/torch combo
        # doesn't support sdpa for Qwen2, so this never becomes a new crash.
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
                **kw,
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="eager",
                **kw,
            )
        _model.eval()
    return _model, _tokenizer


def warmup_llm():
    """Load model into GPU memory for instant subsequent generation."""
    try:
        get_model()
        return _model is not None
    except Exception:
        return False


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False

def start_background_warmup():
    """
    Kicks off model loading in a background thread exactly once per process,
    called at app.py import time. This way the model is already warm -- or
    already warming up -- before anyone opens the AI Copilot tab, instead of
    blocking on someone's first click mid-demo. get_model()'s _load_lock means
    a manual warmup_llm() call or a real chat request made while this thread
    is still loading just waits for it, rather than starting a second,
    duplicate (and GPU-memory-doubling) load.
    """
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Core low-overhead generation helper."""
    model, tok = get_model()
    tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(tmpl, return_tensors="pt").to(model.device)
    gen_kw = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if greedy:
        gen_kw["do_sample"] = False
    else:
        gen_kw["do_sample"]    = True
        gen_kw["temperature"]  = 0.2
        gen_kw["top_p"]        = 0.9
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, schema_keys=None):
    """Returns a structured JSON dict from the model — greedy, minimal tokens."""
    sys_p = "You are an AI logistics engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150,
        greedy=True,
    )
    def _repair_json(text):
        text = re.sub(r'```json\s*|\s*```', '', text)
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: text = m.group(0)
        # Fix missing commas between key-value pairs (e.g. "val"\n"key": or "val" "key":)
        text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
        text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
        # Fix trailing commas before closing brace
        text = re.sub(r',\s*\}', '}', text)
        return text

    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            # Fallback regex extraction of key-value pairs if strict JSON still fails
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km: out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else: out[k] = "N/A"
            if any(v != "N/A" for v in out.values()): return out
        return {"error": "JSON parse failed", "raw": raw}


# ── Agent Roles ───────────────────────────────────────────────────────────────
AGENT_ROLES = {
    "agent1": ("Global Pricing & Port Congestion Agent",
               "You specialise in base freight rates, fuel indexes, and port congestion surcharges."),
    "agent2": ("Route Optimization & Marine Weather Agent",
               "You specialise in shipping route delays, marine weather disruptions, and dwell times."),
    "agent3": ("Carrier Audit & Tariff Compliance Agent",
               "You specialise in carrier punctuality, fuel surcharges, and customs tariff compliance."),
}


# ── Rule-Based Fallback (Section 8) ─────────────────────────────────────────
# Used whenever the LLM is still warming up / unavailable, so the Copilot
# ALWAYS answers immediately instead of the UI hanging while it waits on
# get_model()'s load lock. This is expected behavior per the instructions,
# not a bug — once the model finishes loading, subsequent calls use it.
def rule_based_answer(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Deterministic, data-driven fallback answer — no GPU/model required."""
    a1, a2, a3 = agent1_context or {}, agent2_context or {}, agent3_context or {}
    cong        = a1.get("congestion", "Moderate")
    fuel        = a1.get("fuel_surcharge_pct", 0)
    delay_risk  = a2.get("delay_risk_pct", 0)
    canal       = a2.get("canal_queue", False)
    carrier     = a3.get("carrier", "the selected carrier")
    punctuality = a3.get("punctuality", 0)
    compliance  = a3.get("compliance", "Unknown")
    parts = [
        f"Port congestion is currently **{cong}** with a fuel surcharge of **{fuel}%**, "
        f"which is pushing freight costs upward.",
        f"Route risk stands at **{delay_risk}%** delay probability"
        + (" and canal queues are active" if canal else "") + ".",
        f"**{carrier}** has a punctuality score of **{punctuality*100:.0f}%** and "
        f"compliance status **{compliance}**.",
        "Recommendation: monitor congestion trends and confirm carrier compliance before finalizing the quote.",
    ]
    return " ".join(parts)


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Single-pass structured generation — outputs Agent 1 / Agent 2 / Agent 3 views
    and Executive Synthesis simultaneously. Target latency: ~2 sec on T4.
    """
    if not is_llm_loaded():
        start_background_warmup()
        fallback_synth = ("_(LLM still warming up — rule-based synthesis below.)_\n\n"
                          + rule_based_answer(user_query, agent1_context, agent2_context, agent3_context, db_stats))
        return {
            "agent1": "Port congestion and fuel surcharges are driving cost upward.",
            "agent2": "Marine weather and dwell times pose moderate delay risk.",
            "agent3": "Carrier compliance metrics are within acceptable thresholds.",
            "synthesis": fallback_synth,
        }
    system_prompt = (
        "You are the FreightQuote AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on pricing/congestion>\n"
        "[AGENT 2]: <1 bullet on route/weather>\n"
        "[AGENT 3]: <1 bullet on carrier audit>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100,
        greedy=True,
    )
    res = {
        "agent1": "Port congestion and fuel surcharges are driving cost upward.",
        "agent2": "Marine weather and dwell times pose moderate delay risk.",
        "agent3": "Carrier compliance metrics are within acceptable thresholds.",
        "synthesis": raw,
    }
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Fast greedy single-pass answer — target latency ~1.5 sec on T4.
    NON-BLOCKING: if the LLM isn't loaded/warmed yet, returns the rule-based
    fallback immediately instead of making the user wait on the model load."""
    if not is_llm_loaded():
        start_background_warmup()
        return ("_(LLM still warming up in the background — showing rule-based "
                 "analysis; ask again in a moment for the full model response.)_\n\n"
                 + rule_based_answer(user_question, agent1_context, agent2_context, agent3_context, db_stats))
    sys_p = (
        "You are FreightQuote AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    try:
        return _run(
            [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
            max_tokens=90,
            greedy=True,
        )
    except Exception as e:
        return f"⚠️ LLM generation failed ({e}). Rule-based analysis:\n\n" + rule_based_answer(
            user_question, agent1_context, agent2_context, agent3_context, db_stats)


Writing llm_engine.py


In [10]:
%%writefile config.py
"""
config.py — FreightQuote AI (v3 FINAL)
All secrets from Colab userdata. No hardcoded credentials anywhere.
"""
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

try:
    from __main__ import (STORAGE_DIR, NGROK_AUTHTOKEN, HF_TOKEN,
                          KAGGLE_USERNAME, KAGGLE_KEY, EMAIL_PASSWORD,
                          ADMIN_EMAIL, ADMIN_PASSWORD, EMAIL_ID)
except ImportError:
    STORAGE_DIR    = ("/content/drive/MyDrive/FreightQuote_AI"
                      if os.path.exists("/content/drive/MyDrive") else
                      os.path.abspath("./data/FreightQuote_AI"))
    NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
    NGROK_AUTH_TOKEN = NGROK_AUTHTOKEN # Alias for launch cell compatibility
    HF_TOKEN        = _get_secret("HF_TOKEN")
    KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
    EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
    EMAIL_ID        = _get_secret("EMAIL_ID")
    JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "freightquote-dev-secret-changeme"
    ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID")  or "infosys@ai"
    ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD")  or "admin@123"

os.makedirs(STORAGE_DIR, exist_ok=True)
DB_PATH          = os.path.join(STORAGE_DIR, "freightquote.db")
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "pricing_rf.joblib")
AGENT2_MODEL_PATH = os.path.join(MODELS_DIR, "delay_risk_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "carrier_audit_gb.joblib")


Writing config.py


In [11]:
%%writefile ui_theme.py
"""
Shared ui_theme.py for FreightQuote AI & FranchiseOps AI
Classic Navy & Gold professional styling (matches Milestone 1), layout cards, and status badges.
"""
import streamlit as st

COLORS = {
    "navy_deep":     "#0b1530",
    "navy":          "#0f1c3f",
    "navy_light":    "#1c2e5c",
    "gold":          "#c9a24b",
    "gold_hover":    "#b8912f",
    "gold_light":    "#e8d9ad",
    "bg_main":       "#f4f5f7",
    "bg_card":       "#ffffff",
    "bg_alt":        "#eef0f4",
    "text_heading":  "#0b1530",
    "text_body":     "#1b2436",
    "text_main":     "#1b2436",
    "text_muted":    "#5b6478",
    "text_on_navy":  "#f4f5f7",
    "border":        "#d8dbe2",
    "accent":        "#c9a24b",
    "accent_subtle": "#e8d9ad",
    "accent_text":   "#0b1530",
    "cyan":          "#e3f6f5",
    "pink":          "#ffd3e2",
    "green":         "#2f8f5b",
    "yellow":        "#c9a24b",
    "red":           "#b3413a",
}

NEO_BRUTALIST_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700;800&family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@500;700&display=swap');

html, body, [class*="css"] {{
    font-family: 'Inter', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]};
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Playfair Display', serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
    letter-spacing: 0.3px;
}}

.pn-card {{
    background: {COLORS["bg_card"]};
    border: 1px solid {COLORS["border"]};
    border-top: 3px solid {COLORS["gold"]};
    border-radius: 10px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 0 2px 10px rgba(11,21,48,0.06);
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}}
.pn-card:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 16px rgba(11,21,48,0.10);
}}
.pn-card-alt {{
    background: {COLORS["cyan"]};
    border: 1px solid {COLORS["border"]};
    border-top: 3px solid {COLORS["gold"]};
    border-radius: 10px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 0 2px 10px rgba(11,21,48,0.06);
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 12px;
    border: 1.5px solid {COLORS["border"]};
    border-radius: 6px;
    font-family: 'JetBrains Mono', monospace;
    font-weight: 700;
    font-size: 13px;
    text-transform: uppercase;
}}
.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: {COLORS["navy"]};
    color: {COLORS["gold_light"]};
    border: 1px solid {COLORS["gold"]};
    border-radius: 8px;
    font-family: 'Playfair Display', serif;
    font-weight: 700;
    font-size: 14px;
}}

/* Streamlit Buttons Matching Login Portal (Navy + Gold) */
div.stButton > button {{
    background: {COLORS["navy"]} !important;
    color: {COLORS["gold_light"]} !important;
    font-family: 'Inter', sans-serif !important;
    font-weight: 600 !important;
    border: 1px solid {COLORS["navy"]} !important;
    border-radius: 6px !important;
    padding: 10px 22px !important;
    letter-spacing: 0.4px;
    transition: all 0.2s ease !important;
}}
div.stButton > button:hover {{
    background: {COLORS["gold"]} !important;
    color: {COLORS["navy_deep"]} !important;
    border-color: {COLORS["gold"]} !important;
}}

/* Streamlit Inputs & Selectboxes Matching Login Portal */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div {{
    background: {COLORS["bg_card"]} !important;
    border: 1.5px solid {COLORS["border"]} !important;
    border-radius: 6px !important;
}}
div[data-baseweb="input"]:focus-within {{
    border-color: {COLORS["gold"]} !important;
    box-shadow: 0 0 0 3px rgba(201,162,75,0.18) !important;
}}

/* Streamlit Tabs Matching Login Portal */
button[data-baseweb="tab"] {{
    font-family: 'Inter', sans-serif !important;
    font-weight: 600 !important;
    color: {COLORS["text_muted"]} !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: {COLORS["text_heading"]} !important;
    border-bottom: 3px solid {COLORS["gold"]} !important;
}}

/* Sidebar — Navy gradient like Milestone 1 */
section[data-testid="stSidebar"] {{
    background: linear-gradient(180deg, {COLORS["navy"]} 0%, {COLORS["navy_deep"]} 100%) !important;
    border-right: 1px solid {COLORS["gold"]} !important;
}}
section[data-testid="stSidebar"] * {{
    color: {COLORS["text_on_navy"]} !important;
}}

h1.fq-banner-title {{ color: #ffffff !important; }}
</style>
"""

def inject_css():
    st.markdown(NEO_BRUTALIST_CSS, unsafe_allow_html=True)

def apply_theme():
    inject_css()

def render_header(title, subtitle="", icon="\U0001F3DB\uFE0F"):
    inject_css()
    st.markdown(f"""
    <div style="background:linear-gradient(90deg, {COLORS['navy_deep']} 0%, {COLORS['navy']} 100%);border-radius:12px;padding:22px 28px;margin-bottom:24px;border-bottom:3px solid {COLORS['gold']};">
        <div style="display:flex;align-items:center;gap:16px;">
            <div style="font-size:38px;line-height:1;">{icon}</div>
            <div>
                <h1 class="fq-banner-title" style="margin:0;font-size:24px;letter-spacing:-0.3px;">{title}</h1>
                <p style="margin:4px 0 0;color:#b7bfd6;font-size:14px;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)

def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"], "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:{c};">{text}</span>'


Writing ui_theme.py


In [12]:
%%writefile auth.py
"""
FreightQuote AI - auth.py
Standardized SQLite authentication system matching Login_Page (1).ipynb.
Supports Login (with progressive lockout), Register (with password-strength
checker), Forgot Password via Security Question OR Email OTP (with resend
cooldown), and JWT tokens.
"""
import sqlite3, jwt, bcrypt, datetime, time, random, smtplib
from email.mime.text import MIMEText
import streamlit as st
try:
    from config import DB_PATH, JWT_SECRET_KEY
    JWT_SECRET = JWT_SECRET_KEY
except (ImportError, AttributeError):
    from config import DB_PATH
    JWT_SECRET = "super-secret-freightquote-key-2026"
try:
    from config import EMAIL_ID, EMAIL_PASSWORD
except (ImportError, AttributeError):
    EMAIL_ID, EMAIL_PASSWORD = "", ""
from ui_theme import COLORS

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    try: return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except: return False

def make_jwt(email, username):
    return jwt.encode({"email": email, "username": username, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6)}, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

# ── Password strength (Section 6) ──────────────────────────────────────────
def password_strength(pw):
    """Returns (level, badge_emoji, message, blocked)"""
    n = len(pw)
    if n < 5:
        return "Weak", "🔴", "Password too weak (minimum 5 characters required).", True
    elif n < 10:
        return "Average", "🟡", "Average strength (10+ characters recommended for enterprise security).", False
    else:
        return "Good", "🟢", "Good password strength — proceed with bcrypt hashing.", False

def render_password_strength(pw, key_prefix=""):
    if not pw:
        return None
    level, emoji, msg, blocked = password_strength(pw)
    color = {"Weak": COLORS["red"], "Average": COLORS["yellow"], "Good": COLORS["green"]}[level]
    st.markdown(f'<div style="padding:6px 10px;border-radius:6px;background:{color}22;'
                f'border-left:4px solid {color};font-size:13px;margin:4px 0;">'
                f'{emoji} <b>{level}</b> — {msg}</div>', unsafe_allow_html=True)
    return level, blocked

# ── OTP email sending ────────────────────────────────────────────────────────
def send_otp_email(to_email, otp):
    if not (EMAIL_ID and EMAIL_PASSWORD):
        st.warning(f"📧 (Console fallback — no EMAIL_ID/EMAIL_PASSWORD secret set) Your OTP is: **{otp}**")
        print(f"[OTP FALLBACK] {to_email} -> {otp}")
        return True
    try:
        msg = MIMEText(f"Your Infosys Freight Quote Portal verification code is: {otp}\n\nThis code expires in 5 minutes.")
        msg["Subject"] = "Infosys Freight Quote Portal - Password Reset OTP"
        msg["From"] = EMAIL_ID
        msg["To"] = to_email
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(EMAIL_ID, EMAIL_PASSWORD)
            server.sendmail(EMAIL_ID, to_email, msg.as_string())
        return True
    except Exception as e:
        st.error(f"Email send failed: {e}")
        return False

@st.cache_resource
def init_auth():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT,
            role TEXT DEFAULT 'User',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        for col, coltype in [
            ("security_question", "TEXT"), ("security_answer_hash", "TEXT"),
            ("failed_attempts", "INTEGER DEFAULT 0"),
            ("lock_until", "TIMESTAMP DEFAULT NULL"),
            ("account_status", "TEXT DEFAULT 'active'"),
        ]:
            try: conn.execute(f"ALTER TABLE users ADD COLUMN {col} {coltype}")
            except Exception: pass
        if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
            conn.execute("""INSERT OR IGNORE INTO users
                         (username, email, password_hash, security_question, security_answer_hash, role, account_status)
                         VALUES (?, ?, ?, ?, ?, ?, ?)""",
                         ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin"), "Admin", "active"))
            conn.commit()

# ── Progressive lockout (Section 5) ──────────────────────────────────────────
def _now():
    return datetime.datetime.utcnow()

def register_failed_login(email):
    """Increment failed_attempts and apply lockout tiers. Returns a user-facing message or None."""
    with get_conn() as conn:
        row = conn.execute("SELECT failed_attempts FROM users WHERE email=? OR username=?", (email, email)).fetchone()
        if not row:
            return None
        attempts = (row[0] or 0) + 1
        msg = None
        lock_until = None
        status = "active"
        if attempts == 3:
            lock_until = _now() + datetime.timedelta(seconds=300)
            msg = "⏳ Account temporarily locked for 5 minutes due to 3 failed attempts."
        elif attempts == 4:
            lock_until = _now() + datetime.timedelta(seconds=900)
            msg = "⏳ Account temporarily locked for 15 minutes due to 4 failed attempts."
        elif attempts >= 5:
            status = "locked"
            lock_until = None
            msg = "❌ Account permanently locked due to 5 failed attempts. Only the System Administrator can unlock this account via the Admin Dashboard."
        conn.execute("UPDATE users SET failed_attempts=?, lock_until=?, account_status=? WHERE email=? OR username=?",
                     (attempts, lock_until, status, email, email))
        conn.commit()
        return msg

def reset_failed_login(email):
    with get_conn() as conn:
        conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE email=? OR username=?",
                     (email, email))
        conn.commit()

def check_lock_status(email):
    """Returns (is_locked: bool, message: str|None). Auto-clears expired temporary locks."""
    with get_conn() as conn:
        row = conn.execute("SELECT account_status, lock_until FROM users WHERE email=? OR username=?", (email, email)).fetchone()
    if not row:
        return False, None
    status, lock_until = row
    if status == "locked":
        return True, "❌ Account permanently locked due to 5 failed attempts. Only the System Administrator can unlock this account via the Admin Dashboard."
    if lock_until:
        try:
            lu = datetime.datetime.fromisoformat(str(lock_until))
        except Exception:
            lu = None
        if lu and _now() < lu:
            remaining = int((lu - _now()).total_seconds())
            mins, secs = divmod(remaining, 60)
            return True, f"⏳ Account temporarily locked. Try again in {mins}m {secs}s."
        elif lu:
            reset_failed_login(email)
    return False, None

# ── OTP resend rate limiting (Section 5.1) ───────────────────────────────────
def _otp_cooldown_seconds(resend_count):
    return {0: 0, 1: 60, 2: 180, 3: 300}.get(resend_count, 3600)

def _otp_cooldown_message(resend_count):
    return {
        1: "⏳ Please wait 60 seconds before requesting another OTP.",
        2: "⏳ Please wait 3 minutes before requesting another OTP.",
        3: "⏳ Please wait 5 minutes before requesting another OTP.",
    }.get(resend_count, "⚠️ Too many OTP requests. Please wait 1 hour before trying again.")

def render_auth_portal():
    init_auth()
    if "token" not in st.session_state: st.session_state["token"] = None
    if "auth_tab" not in st.session_state: st.session_state["auth_tab"] = "Login"
    for k, v in [("reset_email", None), ("reset_q", None), ("otp_code", None),
                 ("otp_expiry", None), ("otp_sent_to", None), ("otp_resend_count", 0),
                 ("otp_next_allowed", 0)]:
        if k not in st.session_state: st.session_state[k] = v

    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:44px;margin-bottom:8px;">🏛️</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Freight Quote Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">Enterprise Multi-Agent Logistics & Pricing System</p>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns([1, 2, 1])
    with c2:
        tab1, tab2, tab3 = st.tabs(["🔐 Sign In", "📝 Register Account", "🔑 Reset Password"])

        # ── TAB 1: LOGIN with progressive lockout ─────────────────────────────
        with tab1:
            login_email = st.text_input("Email / Username", key="l_email", placeholder="infosys@ai")
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login"):
                is_locked, lock_msg = check_lock_status(login_email)
                if is_locked:
                    st.error(lock_msg)
                else:
                    with get_conn() as conn:
                        user = conn.execute("SELECT username, email, password_hash, role FROM users WHERE email=? OR username=?", (login_email, login_email)).fetchone()
                    if user and check_txt(login_pw, user[2]):
                        reset_failed_login(login_email)
                        st.session_state["token"] = make_jwt(user[1], user[0])
                        st.session_state["username"] = user[0]
                        st.session_state["role"] = user[3]
                        st.success(f"Welcome back, {user[0]} [{user[3]}]!")
                        st.rerun()
                    else:
                        fail_msg = register_failed_login(login_email) if user else None
                        st.error(fail_msg or "Invalid email/username or password.")

        # ── TAB 2: REGISTER with password strength checker ────────────────────
        with tab2:
            r_user = st.text_input("Username", key="r_u")
            r_email = st.text_input("Email Address", key="r_e")
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            strength_result = render_password_strength(r_pw, key_prefix="r")
            r_pw2 = st.text_input("Confirm Password", type="password", key="r_p2")
            if r_pw2:
                if r_pw2 == r_pw:
                    st.markdown(f'<div style="padding:6px 10px;border-radius:6px;background:{COLORS["green"]}22;'
                                f'border-left:4px solid {COLORS["green"]};font-size:13px;margin:4px 0;">'
                                f'✅ Passwords match.</div>', unsafe_allow_html=True)
                else:
                    st.markdown(f'<div style="padding:6px 10px;border-radius:6px;background:{COLORS["red"]}22;'
                                f'border-left:4px solid {COLORS["red"]};font-size:13px;margin:4px 0;">'
                                f'❌ Passwords do not match.</div>', unsafe_allow_html=True)
            r_role = st.selectbox("Select Enterprise Role", ["Logistics Manager", "Pricing Analyst", "Carrier Auditor", "Executive"], key="r_role")
            r_q = st.selectbox("Security Question", ["What is your pet name?", "What city were you born in?", "What is your favorite school teacher's name?"], key="r_q")
            r_a = st.text_input("Security Answer", key="r_a")
            if st.button("✨ Create Enterprise Account", key="btn_reg"):
                if r_pw and password_strength(r_pw)[3]:
                    st.warning(password_strength(r_pw)[2])
                elif r_pw != r_pw2:
                    st.error("❌ Passwords do not match. Please re-enter Confirm Password.")
                elif r_user and r_email and r_pw and r_pw2 and r_a:
                    try:
                        with get_conn() as conn:
                            conn.execute("INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role, account_status) VALUES (?, ?, ?, ?, ?, ?, 'active')",
                                         (r_user, r_email, hash_txt(r_pw), r_q, hash_txt(r_a.lower().strip()), r_role))
                            conn.commit()
                        st.success(f"Account registered with role [{r_role}]! Please switch to Sign In tab.")
                    except Exception:
                        st.error("Registration failed: Email or username may already exist.")
                else:
                    st.warning("Please fill out all fields.")

        # ── TAB 3: FORGOT PASSWORD — Security Question OR Email OTP ──────────
        with tab3:
            if not st.session_state["reset_email"]:
                f_email = st.text_input("Registered Email", key="f_e")
                colA, colB = st.columns(2)
                if colA.button("🔒 Via Security Question", key="btn_sq"):
                    with get_conn() as conn:
                        u = conn.execute("SELECT security_question FROM users WHERE email=?", (f_email,)).fetchone()
                    if u:
                        st.session_state["reset_email"] = f_email
                        st.session_state["reset_q"] = u[0]
                        st.session_state["reset_mode"] = "sq"
                        st.rerun()
                    else:
                        st.error("Email not found.")
                if colB.button("📧 Via OTP", key="btn_otp"):
                    with get_conn() as conn:
                        u = conn.execute("SELECT id FROM users WHERE email=?", (f_email,)).fetchone()
                    if not u:
                        st.error("Email not found.")
                    else:
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(f_email, otp):
                            st.session_state["otp_code"] = otp
                            st.session_state["otp_expiry"] = time.time() + 300
                            st.session_state["otp_sent_to"] = f_email
                            st.session_state["reset_email"] = f_email
                            st.session_state["reset_mode"] = "otp"
                            st.session_state["otp_resend_count"] = 0
                            st.success(f"✅ OTP sent to {f_email}")
                            time.sleep(1)
                            st.rerun()

            elif st.session_state.get("reset_mode") == "sq":
                st.info(f"Security Question: **{st.session_state.get('reset_q')}**")
                ans_try = st.text_input("Enter Answer", key="f_ans")
                new_pw = st.text_input("New Password", type="password", key="f_npw")
                render_password_strength(new_pw, key_prefix="f")
                if st.button("Confirm Password Reset", key="btn_f2"):
                    if new_pw and password_strength(new_pw)[3]:
                        st.warning(password_strength(new_pw)[2])
                    else:
                        with get_conn() as conn:
                            u_hash = conn.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state["reset_email"],)).fetchone()
                        if u_hash and check_txt(ans_try.lower().strip(), u_hash[0]):
                            with get_conn() as conn:
                                conn.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(new_pw), st.session_state["reset_email"]))
                                conn.commit()
                            st.success("Password reset successfully! Please sign in.")
                            st.session_state["reset_email"] = None
                        else:
                            st.error("Incorrect security answer.")
                if st.button("← Cancel", key="btn_cancel_sq"):
                    st.session_state["reset_email"] = None
                    st.rerun()

            elif st.session_state.get("reset_mode") == "otp":
                st.info(f"📧 A 6-digit code was sent to **{st.session_state['otp_sent_to']}**. It expires in 5 minutes.")
                entered_otp = st.text_input("Enter OTP", max_chars=6, key="f_otp")
                new_pw = st.text_input("New Password", type="password", key="f_npw_otp")
                render_password_strength(new_pw, key_prefix="fo")
                colV, colR = st.columns(2)
                if colV.button("✅ Verify & Reset", key="btn_verify_otp"):
                    if time.time() > (st.session_state["otp_expiry"] or 0):
                        st.error("❌ OTP expired. Please request a new one.")
                    elif entered_otp != st.session_state["otp_code"]:
                        st.error("❌ Incorrect OTP.")
                    elif new_pw and password_strength(new_pw)[3]:
                        st.warning(password_strength(new_pw)[2])
                    else:
                        with get_conn() as conn:
                            conn.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(new_pw), st.session_state["reset_email"]))
                            conn.commit()
                        st.success("Password reset successfully! Please sign in.")
                        st.session_state["reset_email"] = None
                        st.session_state["otp_code"] = None
                if colR.button("🔁 Resend OTP", key="btn_resend_otp"):
                    if time.time() < st.session_state["otp_next_allowed"]:
                        remaining = int(st.session_state["otp_next_allowed"] - time.time())
                        st.warning(_otp_cooldown_message(st.session_state["otp_resend_count"]) + f" ({remaining}s left)")
                    else:
                        st.session_state["otp_resend_count"] += 1
                        cooldown = _otp_cooldown_seconds(st.session_state["otp_resend_count"])
                        st.session_state["otp_next_allowed"] = time.time() + cooldown
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(st.session_state["reset_email"], otp):
                            st.session_state["otp_code"] = otp
                            st.session_state["otp_expiry"] = time.time() + 300
                            st.success(_otp_cooldown_message(st.session_state["otp_resend_count"] - 1) if st.session_state["otp_resend_count"] > 1 else "✅ New OTP sent.")
                if st.button("← Cancel", key="btn_cancel_otp"):
                    st.session_state["reset_email"] = None
                    st.session_state["otp_code"] = None
                    st.rerun()


Writing auth.py


In [13]:
%%writefile db.py
import sqlite3
from config import DB_PATH

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY, carrier_name TEXT, transport_mode TEXT,
            punctuality_rate REAL, avg_delay_days REAL, fuel_surcharge_pct REAL,
            tariff_compliance_score REAL, tier_rating TEXT, flagged INTEGER DEFAULT 0)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS quotes (
            quote_id TEXT PRIMARY KEY, created_by TEXT, origin TEXT, destination TEXT,
            distance_nm REAL, weight_tons REAL, shipment_mode TEXT, port_congestion TEXT,
            cargo_type TEXT, base_cost_usd REAL, margin_usd REAL, adjustment_factor REAL,
            final_cost_usd REAL, delay_risk_prob REAL, risk_summary TEXT, audit_flag TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY, quote_id TEXT, carrier_name TEXT,
            actual_cost REAL, transit_days INTEGER, delay_days INTEGER,
            status TEXT DEFAULT 'In Transit',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            origin TEXT, destination TEXT, distance_nm REAL, weight_tons REAL,
            freight_cost_usd REAL, shipment_mode TEXT, port_congestion TEXT,
            dwell_time_days REAL, berth_capacity INTEGER, weather_disruption_level REAL,
            carrier_punctuality REAL, fuel_surcharge_pct REAL, compliance_status TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE,
            email TEXT UNIQUE, password_hash TEXT,
            security_question TEXT, security_answer_hash TEXT,
            role TEXT DEFAULT 'User',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        try: conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        except Exception: pass
        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()

def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role":r[0],"content":r[1]} for r in reversed(rows)]

def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()

def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()


Writing db.py


In [14]:
%%writefile weather_context.py
"""
weather_context.py for FreightQuote AI
Simulates Indian marine ports and global trade route weather conditions.
"""
import random

GLOBAL_PORTS_WEATHER = {
    "Mumbai JNPT (IN)": {"status": "Monsoon Rain & High Winds", "temp_c": 28, "wind_kt": 32, "delay_penalty_multiplier": 1.15},
    "Mundra Port (IN)": {"status": "Clear / Dusty Gusts", "temp_c": 34, "wind_kt": 18, "delay_penalty_multiplier": 1.05},
    "Chennai Port (IN)": {"status": "Tropical Cyclone Watch", "temp_c": 31, "wind_kt": 36, "delay_penalty_multiplier": 1.20},
    "Cochin Port (IN)": {"status": "Monsoon Squalls", "temp_c": 27, "wind_kt": 24, "delay_penalty_multiplier": 1.10},
    "Kolkata Haldia (IN)": {"status": "Heavy River Fog & Tidal Delay", "temp_c": 26, "wind_kt": 14, "delay_penalty_multiplier": 1.12},
    "Shanghai (CN)": {"status": "High Winds & Typhoon Watch", "temp_c": 22, "wind_kt": 38, "delay_penalty_multiplier": 1.18},
    "Rotterdam (NL)": {"status": "Clear / Moderate Gale", "temp_c": 14, "wind_kt": 22, "delay_penalty_multiplier": 1.05},
    "Singapore (SG)": {"status": "Monsoon Rain Squalls", "temp_c": 29, "wind_kt": 26, "delay_penalty_multiplier": 1.08},
    "Suez Canal Hub": {"status": "Sandstorm & High Transit Queue", "temp_c": 35, "wind_kt": 30, "delay_penalty_multiplier": 1.25},
    "Panama Canal Hub": {"status": "Drought Water Level Restrictions", "temp_c": 31, "wind_kt": 15, "delay_penalty_multiplier": 1.30},
    "Dubai (AE)": {"status": "Clear / High Heat", "temp_c": 38, "wind_kt": 14, "delay_penalty_multiplier": 1.02},
    "Hamburg (DE)": {"status": "Heavy Fog & Berth Queue", "temp_c": 11, "wind_kt": 18, "delay_penalty_multiplier": 1.12}
}

def get_weather_report(port_name):
    for k, v in GLOBAL_PORTS_WEATHER.items():
        if k.lower() in port_name.lower() or port_name.lower() in k.lower():
            return {"port": k, **v}
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}

def get_route_weather_multiplier(origin, dest):
    w1 = get_weather_report(origin)
    w2 = get_weather_report(dest)
    return round((w1["delay_penalty_multiplier"] + w2["delay_penalty_multiplier"]) / 2, 3)

def get_city_weather(city_name):
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}


Writing weather_context.py


In [15]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


Writing notifications.py


In [16]:
%%writefile seed_data.py
"""
FreightQuote AI - seed_data.py
Pre-seeds the database with realistic global carriers, quotes, shipments, and merged Kaggle tables.
"""
from db import get_conn, init_db
from notifications import send_alert

def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Carriers
        if not conn.execute("SELECT count(*) FROM carriers").fetchone()[0]:
            carriers = [
                ("CAR-001", "Maersk Global Line", "Ocean", 0.94, 1.2, 12.5, 0.98, "Tier 1 (Apex)"),
                ("CAR-002", "MSC Mediterranean Shipping", "Ocean", 0.91, 1.8, 13.0, 0.96, "Tier 1 (Apex)"),
                ("CAR-003", "CMA CGM Logistics", "Ocean", 0.88, 2.4, 14.2, 0.92, "Tier 2 (Standard)"),
                ("CAR-004", "DHL Air Cargo Express", "Air", 0.99, 0.2, 18.0, 0.99, "Tier 1 (Apex)"),
                ("CAR-005", "FedEx International Freight", "Air", 0.98, 0.3, 17.5, 0.99, "Tier 1 (Apex)"),
                ("CAR-006", "DB Schenker Overland Rail", "Rail/Truck", 0.89, 2.1, 11.0, 0.94, "Tier 2 (Standard)"),
            ]
            conn.executemany("INSERT INTO carriers (carrier_id, carrier_name, transport_mode, "
            "punctuality_rate, avg_delay_days, fuel_surcharge_pct, "
            "tariff_compliance_score, tier_rating) VALUES (?, ?, ?, ?, ?, ?, ?, ?)", carriers)

        # Seed Quotes
        if not conn.execute("SELECT count(*) FROM quotes").fetchone()[0]:
            quotes = [
                ("Q-1001", "infosys@ai", "Mumbai JNPT (IN)", "Rotterdam (NL)", 10500, 45.0, "Ocean", "High", "Electronics", 18500, 3200, 1.15, 24304, 0.96, "Moderate Risk (Monsoon)", "Passed Audit"),
                ("Q-1002", "infosys@ai", "Shanghai (CN)", "Mundra Port (IN)", 7800, 120.0, "Ocean", "Medium", "General Cargo", 42000, 4500, 1.05, 48360, 0.95, "Low Risk", "Passed Audit"),
                ("Q-1003", "infosys@ai", "Chennai Port (IN)", "Singapore (SG)", 4800, 15.0, "Air", "Low", "Pharmaceuticals", 31000, 0, 1.08, 33170, 0.98, "Minimal Risk", "Passed Audit"),
                ("Q-1004", "infosys@ai", "Cochin Port (IN)", "Dubai (AE)", 10800, 60.0, "Ocean", "High", "Chemicals", 26000, 5200, 1.12, 35880, 0.94, "High Risk (Squalls)", "Flagged Surcharge"),
            ]
            conn.executemany("INSERT INTO quotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)", quotes)

        # Seed Shipments
        if not conn.execute("SELECT count(*) FROM shipments").fetchone()[0]:
            shipments = [
                ("SH-8001", "Q-1001", "Maersk Global Line", 24304, 32, 2, "Delivered"),
                ("SH-8002", "Q-1002", "MSC Mediterranean Shipping", 48360, 24, 0, "Delivered"),
                ("SH-8003", "Q-1003", "DHL Air Cargo Express", 33170, 3, 0, "In Transit"),
                ("SH-8004", "Q-1004", "CMA CGM Logistics", 35880, 35, 5, "Delayed (Port Queue)"),
            ]
            conn.executemany("INSERT INTO shipments (shipment_id, quote_id, carrier_name, actual_cost, transit_days, delay_days, status) VALUES (?, ?, ?, ?, ?, ?, ?)", shipments)
            conn.commit()

    send_alert("Email", "admin@freightquote.ai", "System Initialized", "Database seeded with 6 carriers, quotes, and historical shipments.")
    print("✅ Database pre-seeded successfully.")


Writing seed_data.py


In [17]:
%%writefile admin_dash.py
"""admin_dash.py — Admin Dashboard for FreightQuote AI (Milestone 2, Section 9)
Clean tabbed layout:
  Tab 1 — User Management  (Add / Delete / Unlock, lockout status)
  Tab 2 — ML Model Card    (R²/RMSE/ROC-AUC per agent, from ml_models table)
  Tab 3 — System & Alerts  (GPU health, LLM activity, live alert log)
Restricted to role == 'Admin' by the caller (app.py).
"""
import subprocess, datetime
import streamlit as st
import pandas as pd
import plotly.express as px
from db import get_conn
from notifications import get_recent_alerts
from ui_theme import render_card, COLORS
from auth import hash_txt

_APP_START = datetime.datetime.now()


def _smi(query):
    try:
        r = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3)
        return r.stdout.strip()
    except Exception:
        return "N/A"


def render_admin_dashboard(project="freight"):
    render_card('<h3 style="margin:0;">🛡️ Admin Dashboard — System Intelligence</h3>')

    tab_users, tab_models, tab_system = st.tabs(
        ["👥 User Management", "📈 ML Model Card", "🔔 System & Alerts"]
    )

    # ══════════════════════════════════════════════════════════════════════
    # TAB 1 — USER MANAGEMENT  (Add / Delete / Unlock)
    # ══════════════════════════════════════════════════════════════════════
    with tab_users:
        with st.expander("➕ Add User", expanded=False):
            with st.form("add_user_form", clear_on_submit=True):
                au1, au2 = st.columns(2)
                new_username = au1.text_input("Username")
                new_email = au2.text_input("Email")
                au3, au4 = st.columns(2)
                new_password = au3.text_input("Initial Password", type="password")
                new_role = au4.selectbox(
                    "Role", ["Admin", "Logistics Manager", "Pricing Analyst",
                             "Carrier Auditor", "Executive"])
                submitted = st.form_submit_button("➕ Create User")
                if submitted:
                    if not (new_username and new_email and new_password):
                        st.warning("Please fill out all fields.")
                    elif len(new_password) < 5:
                        st.warning("🔴 Password too weak (minimum 5 characters required).")
                    else:
                        try:
                            with get_conn() as conn:
                                conn.execute(
                                    "INSERT INTO users (username, email, password_hash, role, account_status) "
                                    "VALUES (?, ?, ?, ?, 'active')",
                                    (new_username, new_email, hash_txt(new_password), new_role))
                                conn.commit()
                            st.success(f"✅ User '{new_username}' created with role [{new_role}].")
                            st.rerun()
                        except Exception:
                            st.error("❌ Could not create user — email or username may already exist.")

        with get_conn() as conn:
            try:
                users_df = pd.read_sql(
                    "SELECT id, username, role, email, created_at, "
                    "COALESCE(failed_attempts,0) AS failed_attempts, "
                    "COALESCE(account_status,'active') AS account_status "
                    "FROM users ORDER BY id DESC", conn)
            except Exception:
                users_df = pd.DataFrame(columns=["id", "username", "role", "email",
                                                  "created_at", "failed_attempts", "account_status"])

        if users_df.empty:
            st.info("No users registered yet.")
        else:
            hdr = st.columns([2, 2, 1.5, 1, 1])
            for c, label in zip(hdr, ["User", "Role", "Status", "Unlock", "Delete"]):
                c.markdown(f'<span style="font-size:12px;color:{COLORS["text_muted"]};font-weight:700;">{label}</span>',
                           unsafe_allow_html=True)
            for _, row in users_df.iterrows():
                is_locked = row.get("account_status") == "locked" or (row.get("failed_attempts", 0) or 0) >= 3
                uc1, uc2, uc3, uc4, uc5 = st.columns([2, 2, 1.5, 1, 1])
                uc1.markdown(f"**{row['username']}**")
                uc2.markdown(f'<span style="color:#0066cc;font-weight:600;">[{row["role"]}]</span>',
                             unsafe_allow_html=True)
                if row.get("account_status") == "locked":
                    uc3.markdown(f'<span style="color:{COLORS["red"]};font-weight:700;">🔒 Locked</span>', unsafe_allow_html=True)
                elif (row.get("failed_attempts", 0) or 0) >= 3:
                    uc3.markdown(f'<span style="color:{COLORS["yellow"]};font-weight:700;">⏳ Temp-locked</span>', unsafe_allow_html=True)
                else:
                    uc3.markdown(f'<span style="color:{COLORS["green"]};font-weight:700;">✅ Active</span>', unsafe_allow_html=True)
                with uc4:
                    if is_locked:
                        if st.button("🔓", key=f"unlock_user_{row['id']}", help=f"Unlock {row['username']}"):
                            with get_conn() as c:
                                c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, "
                                          "account_status='active' WHERE id=?", (row["id"],))
                                c.commit()
                            st.success("✅ User account unlocked successfully.")
                            st.rerun()
                with uc5:
                    if st.button("🗑️", key=f"del_user_{row['id']}", help=f"Delete {row['username']}"):
                        with get_conn() as c:
                            c.execute("DELETE FROM users WHERE id=?", (row["id"],))
                            c.commit()
                        st.success(f"Deleted {row['username']}")
                        st.rerun()

    # ══════════════════════════════════════════════════════════════════════
    # TAB 2 — ML MODEL CARD  (Section 9: R²/RMSE for Pricing, ROC-AUC for others)
    # ══════════════════════════════════════════════════════════════════════
    with tab_models:
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql(
                    "SELECT agent_name, model_name, r2_score, accuracy, "
                    "training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
            except Exception:
                ml_df = pd.DataFrame()

        if ml_df.empty:
            st.info("No model training records found yet. Run training / retrain from the Analytics tab.")
        else:
            for agent_label, agent_key, metric_label in [
                ("💰 Agent 1 — Dynamic Pricing", "Agent1_Pricing", "R² / RMSE"),
                ("🚢 Agent 2 — Route Delay Classifier", "Agent2_DelayRisk", "ROC-AUC"),
                ("✅ Agent 3 — Carrier Compliance Sentinel", "Agent3_CarrierCompliance", "ROC-AUC"),
            ]:
                sub = ml_df[ml_df["agent_name"] == agent_key]
                if sub.empty:
                    continue
                best = sub.loc[sub["r2_score"].astype(float).idxmax()]
                st.markdown(
                    f'<h4 style="color:{COLORS["text_heading"]};margin:14px 0 4px;">{agent_label}</h4>',
                    unsafe_allow_html=True)
                mcol1, mcol2, mcol3 = st.columns(3)
                mcol1.metric("Champion Model", best["model_name"])
                mcol2.metric(metric_label.split(" / ")[0], f'{best["r2_score"]:.4f}')
                if metric_label == "R² / RMSE":
                    mcol3.metric("Algorithms Compared", len(sub))
                else:
                    mcol3.metric("Accuracy", f'{best["accuracy"]*100:.1f}%')
                st.dataframe(sub[["model_name", "r2_score", "accuracy", "training_rows", "created_at"]],
                             use_container_width=True, hide_index=True)

            st.markdown("---")
            st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">📋 Full Metrics Log</h4>',
                        unsafe_allow_html=True)
            st.dataframe(ml_df, use_container_width=True, hide_index=True)

    # ══════════════════════════════════════════════════════════════════════
    # TAB 3 — SYSTEM & ALERTS  (GPU health, LLM activity, live alert log)
    # ══════════════════════════════════════════════════════════════════════
    with tab_system:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">⚙️ System Health</h4>',
                    unsafe_allow_html=True)
        gpu_mem  = _smi("memory.used")
        gpu_tot  = _smi("memory.total")
        gpu_util = _smi("utilization.gpu")
        uptime   = str(datetime.datetime.now() - _APP_START).split(".")[0]
        h1, h2, h3, h4 = st.columns(4)
        for col, icon, label, val in [
            (h1, "🖥️", "GPU VRAM Used",  f"{gpu_mem} / {gpu_tot} MB"),
            (h2, "⚡", "GPU Utilization", f"{gpu_util}%"),
            (h3, "🕒", "App Uptime",      uptime),
            (h4, "✅", "LLM Status",      "Active" if gpu_mem != "N/A" else "Standby"),
        ]:
            col.markdown(
                f'<div class="pn-card" style="text-align:center;padding:14px;">'
                f'<div style="font-size:26px;">{icon}</div>'
                f'<h3 style="margin:6px 0 2px;font-size:1.1rem;">{val}</h3>'
                f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
                f'</div>', unsafe_allow_html=True)

        st.markdown("---")
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🤖 LLM Activity Monitor</h4>',
                    unsafe_allow_html=True)
        with get_conn() as conn:
            try:
                chat_df = pd.read_sql(
                    "SELECT username, count(*) as queries FROM chat_history "
                    "WHERE role='user' GROUP BY username ORDER BY queries DESC", conn)
                total_q = int(chat_df["queries"].sum()) if not chat_df.empty else 0
            except Exception:
                chat_df = pd.DataFrame(columns=["username", "queries"])
                total_q = 0

        mc1, mc2 = st.columns([1, 1.6])
        with mc1:
            st.metric("Total Copilot Queries", total_q)
            st.dataframe(chat_df, use_container_width=True, hide_index=True)
        with mc2:
            if not chat_df.empty:
                fig = px.pie(chat_df, names="username", values="queries",
                             title="Queries per User", hole=0.4,
                             color_discrete_sequence=px.colors.sequential.Teal)
                fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                                  height=250, margin=dict(l=10, r=10, t=40, b=10))
                st.plotly_chart(fig, use_container_width=True)

        st.markdown("---")
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">🔔 Live Alert Log</h4>',
                    unsafe_allow_html=True)
        filt = st.selectbox("Filter by type", ["All", "In-App", "Email", "SMS"], key="admin_alert_filt")
        alerts = get_recent_alerts(50)
        for a in alerts:
            if filt != "All" and a[1].lower() != filt.lower():
                continue
            badge = {"email": "#ffd803", "sms": "#f87171", "in-app": "#34d399"}.get(a[1].lower(), "#bae8e8")
            st.markdown(
                f'<div style="border-left:4px solid {badge};padding:4px 10px;margin:3px 0;'
                f'font-size:13px;"><b>[{a[1].upper()}]</b> {a[3]} '
                f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
                unsafe_allow_html=True)


Writing admin_dash.py


In [18]:
%%writefile agent2_freight.py
"""
agent2_freight.py — Enriched Agent 2: Route Optimization & Marine Weather Risk
New features: Route radar chart, global delay trend, AI advisory, alternative routes table.
Extended ports list covering India, Middle East, Europe, Americas, Asia-Pacific.
"""
import numpy as np
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from ui_theme import render_card, COLORS
from weather_context import get_weather_report
from llm_engine import orchestrate_3_agents_query

# ── Full global port list with Indian ports ───────────────────────────────────
ALL_PORTS = [
    # India
    "Mumbai (IN)", "Chennai (IN)", "Nhava Sheva / JNPT (IN)", "Kolkata (IN)",
    "Mundra (IN)", "Cochin (IN)", "Vishakhapatnam (IN)", "Tuticorin (IN)",
    # China / East Asia
    "Shanghai (CN)", "Shenzhen (CN)", "Ningbo (CN)", "Qingdao (CN)",
    "Tianjin (CN)", "Guangzhou (CN)", "Busan (KR)", "Tokyo (JP)", "Osaka (JP)",
    # South-East Asia
    "Singapore (SG)", "Port Klang (MY)", "Laem Chabang (TH)", "Ho Chi Minh (VN)",
    "Jakarta (ID)",
    # Middle East
    "Dubai / Jebel Ali (AE)", "Abu Dhabi (AE)", "Salalah (OM)", "Dammam (SA)",
    # Europe
    "Rotterdam (NL)", "Hamburg (DE)", "Antwerp (BE)", "Felixstowe (GB)",
    "Barcelona (ES)", "Piraeus (GR)", "Genoa (IT)",
    # Americas
    "Los Angeles (US)", "New York / Newark (US)", "Houston (US)",
    "Santos (BR)", "Buenos Aires (AR)", "Manzanillo (MX)",
    # Africa / Other
    "Durban (ZA)", "Mombasa (KE)", "Port Said (EG)",
    # Canal Hubs
    "Suez Canal Hub", "Panama Canal Hub",
]

# Approximate distances (nm) for common route pairs
_DIST = {
    ("Mumbai (IN)",              "Rotterdam (NL)"):            8600,
    ("Nhava Sheva / JNPT (IN)", "Rotterdam (NL)"):            8700,
    ("Chennai (IN)",             "Singapore (SG)"):            1600,
    ("Mundra (IN)",              "Dubai / Jebel Ali (AE)"):    1050,
    ("Kolkata (IN)",             "Shanghai (CN)"):             3200,
    ("Shanghai (CN)",            "Rotterdam (NL)"):            10500,
    ("Shanghai (CN)",            "Los Angeles (US)"):          6500,
    ("Singapore (SG)",           "Dubai / Jebel Ali (AE)"):   3500,
    ("Singapore (SG)",           "Rotterdam (NL)"):            8300,
    ("Los Angeles (US)",         "Hamburg (DE)"):              7800,
    ("Santos (BR)",              "Rotterdam (NL)"):            5700,
    ("Durban (ZA)",              "Rotterdam (NL)"):            7200,
    ("Busan (KR)",               "Rotterdam (NL)"):            11200,
}

def _dist(o, d):
    return _DIST.get((o, d), _DIST.get((d, o), 7500))


def render_agent2_freight(agent2_m, username, db_stats, a1_ctx, a3_ctx, send_alert, get_conn, confidence_band):
    render_card('<h3 style="margin:0;">🚢 Agent 2: Route Optimization & Marine Weather Risk</h3>')

    c1, c2 = st.columns([1.1, 1])
    with c1:
        origin = st.selectbox("Origin Port", ALL_PORTS, index=0)
        dest   = st.selectbox("Destination Port", ALL_PORTS, index=14)
        dwell  = st.slider("Avg Port Dwell (days)", 0.5, 12.0, 3.5)
        canal  = st.checkbox("Canal Queue Active?", value=True)
        season = st.selectbox("Season / Risk Period",
                              ["Normal","Monsoon (Jun–Sep)","Typhoon Season (Jul–Nov)",
                               "Winter North Sea","Suez Disruption Alert"])

    wo = get_weather_report(origin)
    wd = get_weather_report(dest)
    route_nm = _dist(origin, dest)

    with c2:
        render_card(
            f"<b>📍 Origin:</b> {origin}<br>"
            f"Weather: <b>{wo['status']}</b> | Wind: <b>{wo['wind_kt']} kt</b><br><br>"
            f"<b>📍 Destination:</b> {dest}<br>"
            f"Weather: <b>{wd['status']}</b> | Wind: <b>{wd['wind_kt']} kt</b><br><br>"
            f"<b>🗺️ Route Distance:</b> ~{route_nm:,} nm", alt=True)

        if agent2_m is not None:
            w_avg = (wo["delay_penalty_multiplier"] + wd["delay_penalty_multiplier"]) / 2 - 1.0
            season_risk = {"Normal": 0.20, "Monsoon (Jun–Sep)": 0.55, "Typhoon Season (Jul–Nov)": 0.70,
                           "Winter North Sea": 0.45, "Suez Disruption Alert": 0.65}.get(season, 0.25)
            row = [dwell, 20, float(route_nm), float(w_avg), int(canal), season_risk]
            prob, lo, hi = confidence_band(agent2_m, row)
        else:
            prob = min(0.95, dwell / 12 * 0.5 + (0.15 if canal else 0) +
                       (0.2 if "Typhoon" in season or "Monsoon" in season else 0))
            lo, hi = max(0, prob - 0.08), min(1, prob + 0.08)

        badge_c = "#f87171" if prob > 0.6 else ("#ffd803" if prob > 0.35 else "#34d399")
        st.markdown(
            f'<div style="background:{badge_c};padding:14px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};margin-top:10px;">'
            f'<span class="agent-badge">Agent 2</span>'
            f'<h2 style="color:{COLORS["text_heading"]};margin:6px 0 0;">{prob * 100:.1f}% Delay Risk</h2>'
            f'<p style="margin:4px 0;font-weight:600;">95% CI: {lo * 100:.1f}% — {hi * 100:.1f}%</p>'
            f'<p style="margin:0;font-size:12px;">Season: {season}</p>'
            f'</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_radar, tab_trend, tab_alt, tab_ai = st.tabs(
        ["📡 Route Radar", "📊 Delay Trend", "🔀 Alt Routes", "🤖 AI Advisory"])

    # ── Radar Chart ──────────────────────────────────────────────────────────
    with tab_radar:
        cats = ["Delay Risk", "Congestion Impact", "Weather Severity",
                "Canal Dependency", "Carrier Availability"]
        vals = [
            prob * 10,
            min(10, dwell * 1.2),
            min(10, (wo["wind_kt"] + wd["wind_kt"]) / 15),
            8.0 if canal else 2.0,
            7.5,
        ]
        fig = go.Figure(go.Scatterpolar(r=vals + [vals[0]], theta=cats + [cats[0]],
                                        fill="toself",
                                        line_color=COLORS["accent"],
                                        fillcolor="rgba(0,197,205,0.2)"))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
                          paper_bgcolor="rgba(0,0,0,0)", height=320,
                          margin=dict(l=40, r=40, t=20, b=20))
        st.plotly_chart(fig, use_container_width=True)

    # ── Delay Trend across key routes ────────────────────────────────────────
    with tab_trend:
        routes = [
            "Mumbai→Rotterdam", "Shanghai→Rotterdam", "Singapore→Dubai",
            "LA→Hamburg", "Chennai→Singapore", "Nhava Sheva→Antwerp",
            "Mundra→Jebel Ali", "Kolkata→Shanghai", "Santos→Rotterdam",
        ]
        delays = [62, 68, 38, 55, 28, 58, 22, 45, 48]
        colors = ["#f87171" if d > 55 else ("#ffd803" if d > 35 else "#34d399") for d in delays]
        fig2 = go.Figure(go.Bar(x=routes, y=delays, marker_color=colors,
                                text=[f"{d}%" for d in delays], textposition="outside"))
        fig2.update_layout(title="Delay Probability % — Key Global Routes",
                           paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           yaxis_range=[0, 100], height=320,
                           margin=dict(l=10, r=10, t=40, b=80))
        st.plotly_chart(fig2, use_container_width=True)

    # ── Alternative Routes ────────────────────────────────────────────────────
    with tab_alt:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 10px;">'
                    f'🔀 Alternative Route Suggestions for {origin} → {dest}</h4>',
                    unsafe_allow_html=True)
        alt_data = {
            "Route": [f"{origin} → {dest} (Direct)",
                      f"{origin} → Colombo → {dest}",
                      f"{origin} → Singapore → {dest}"],
            "Extra Distance (nm)": [0, 420, 680],
            "Extra Transit (days)": [0, 1, 2],
            "Risk Level": ["Current", "Lower", "Lowest"],
            "Cost Delta (USD)": [0, "+$380", "+$650"],
        }
        st.dataframe(alt_data, use_container_width=True, hide_index=True)

    # ── AI Advisory ──────────────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Route Advisory", key="btn_a2_advisory"):
            a2_ctx = {"origin": origin, "dest": dest, "dwell": dwell,
                      "canal_queue": canal, "delay_risk_pct": round(prob * 100, 1),
                      "season": season, "route_nm": route_nm}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"Best strategy for {origin} to {dest} route given current conditions?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Route Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Route Advisory", f"{origin}→{dest}")


Writing agent2_freight.py


In [19]:
%%writefile agent3_freight.py
"""
agent3_freight.py — Enriched Agent 3: Carrier Audit & Tariff Compliance
New features: Carrier comparison bar chart, Flag Carrier button, Audit Report generator, Tier Matrix.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from ui_theme import render_card, COLORS
from db import get_conn
from llm_engine import generate_json
from notifications import send_alert


def render_agent3_freight(agent3_m, username, confidence_band):
    render_card('<h3 style="margin:0;">✅ Agent 3: Carrier Audit & Tariff Compliance</h3>')

    with get_conn() as conn:
        carriers_df = pd.read_sql("SELECT * FROM carriers", conn)

    if carriers_df.empty:
        st.warning("No carrier data found. Run seed_data first.")
        return

    # ── Top section: table + audit panel ─────────────────────────────────────
    c1, c2 = st.columns([1.4, 1])
    with c1:
        # Flag badge overlay
        def style_row(row):
            return ["background:#fff0f0" if row.get("flagged", 0) else ""] * len(row)
        st.dataframe(carriers_df, use_container_width=True, hide_index=True)

    with c2:
        sel = st.selectbox("Select Carrier to Audit", carriers_df["carrier_name"].tolist())
        row_c = carriers_df[carriers_df["carrier_name"] == sel].iloc[0]
        complaint = 0.02 if str(row_c.get("tier_rating", "")).lower() == "apex" else 0.06
        X_row = [
            float(row_c["punctuality_rate"]),
            float(row_c["avg_delay_days"]),
            complaint,
            float(row_c["fuel_surcharge_pct"]),
            float(row_c["tariff_compliance_score"]),
            1.0,
        ]
        prob, lo, hi = confidence_band(agent3_m, X_row) if agent3_m else (
            float(row_c["tariff_compliance_score"]), 0.0, 1.0)

        badge_c = "#34d399" if prob > 0.7 else ("#ffd803" if prob > 0.5 else "#f87171")
        is_flagged = bool(row_c.get("flagged", 0))
        st.markdown(
            f'<div style="background:{badge_c};padding:14px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};">'
            f'<span class="agent-badge">Agent 3</span>'
            f'{"<span style=\"background:#f87171;color:#fff;padding:2px 8px;border-radius:6px;font-size:12px;margin-left:8px;\">🚨 FLAGGED</span>" if is_flagged else ""}'
            f'<h2 style="color:{COLORS["text_heading"]};margin:8px 0 0;">{prob * 100:.1f}% Compliance</h2>'
            f'<p style="margin:4px 0;font-weight:600;">95% CI: {lo * 100:.1f}% — {hi * 100:.1f}%</p>'
            f'<p style="margin:0;font-size:12px;">'
            f'Punctuality: {row_c["punctuality_rate"] * 100:.1f}% | '
            f'Fuel: {row_c["fuel_surcharge_pct"]}%</p>'
            f'</div>', unsafe_allow_html=True)

        # Flag / Unflag button
        fa, fb = st.columns(2)
        with fa:
            if st.button("🚨 Flag Carrier" if not is_flagged else "✅ Clear Flag",
                         key="btn_flag", use_container_width=True):
                new_flag = 0 if is_flagged else 1
                with get_conn() as conn:
                    conn.execute("UPDATE carriers SET flagged=? WHERE carrier_name=?",
                                 (new_flag, sel))
                send_alert("In-App", username, "Carrier Flagged" if new_flag else "Flag Cleared", sel)
                st.rerun()
        with fb:
            if st.button("📋 Audit Report", key="btn_report", use_container_width=True):
                with st.spinner("Generating audit report (~2 sec)..."):
                    report = generate_json(
                        f"Carrier: {sel}. Punctuality: {row_c['punctuality_rate']:.2f}. "
                        f"Avg delay: {row_c['avg_delay_days']} days. "
                        f"Tariff compliance: {row_c['tariff_compliance_score']:.2f}. "
                        f"Fuel surcharge: {row_c['fuel_surcharge_pct']}%. "
                        "Generate carrier audit assessment.",
                        schema_keys=["risk_level", "recommended_action",
                                     "penalty_estimate_usd", "next_audit_date"])
                st.json(report)

    st.markdown("---")
    tab_compare, tab_matrix = st.tabs(["📊 Carrier Comparison", "🏆 Tier Matrix"])

    # ── Carrier Comparison Bar Chart ──────────────────────────────────────────
    with tab_compare:
        metrics = st.multiselect(
            "Compare metrics",
            ["punctuality_rate", "tariff_compliance_score", "fuel_surcharge_pct", "avg_delay_days"],
            default=["punctuality_rate", "tariff_compliance_score"])
        if metrics:
            melt = carriers_df[["carrier_name"] + metrics].melt(
                id_vars="carrier_name", var_name="metric", value_name="value")
            fig = px.bar(melt, x="carrier_name", y="value", color="metric", barmode="group",
                         title="Carrier Performance Comparison",
                         color_discrete_sequence=["#00c5cd", "#272343", "#ffd803", "#f87171"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=340, margin=dict(l=10, r=10, t=40, b=80),
                              xaxis_tickangle=-30)
            st.plotly_chart(fig, use_container_width=True)

    # ── Tier Rating Matrix ────────────────────────────────────────────────────
    with tab_matrix:
        carriers_df["composite_score"] = (
            carriers_df["punctuality_rate"] * 0.4 +
            carriers_df["tariff_compliance_score"] * 0.4 +
            (1 - carriers_df["fuel_surcharge_pct"] / 25) * 0.2
        ).round(3)
        ranked = carriers_df[["carrier_name", "tier_rating", "composite_score",
                               "punctuality_rate", "tariff_compliance_score",
                               "fuel_surcharge_pct"]].sort_values(
            "composite_score", ascending=False).reset_index(drop=True)
        ranked.index += 1

        def color_tier(val):
            c = {"Apex": "#d1fae5", "Preferred": "#fef9c3", "Standard": "#fee2e2"}.get(str(val), "")
            return f"background-color:{c}" if c else ""

        st.dataframe(ranked.style.applymap(color_tier, subset=["tier_rating"]),
                     use_container_width=True)


Writing agent3_freight.py


## Step 5 — Initialise Database & Seed Sample Data


In [20]:
import db, seed_data
db.init_db()
seed_data.seed_all()


[EMAIL] To: admin@freightquote.ai | Subject: System Initialized | Status: Delivered
✅ Database pre-seeded successfully.


## Step 6 — Train ML Agents


In [21]:
%%writefile train_ml.py
"""
train_ml.py — FreightQuote AI (v4 — Milestone 2, 6 algorithms per agent)
Multi-Algorithm Comparison:
  Agent 1 (Pricing): RandomForest, GradientBoosting, ExtraTrees, Ridge, DecisionTree, AdaBoost → best R²
  Agent 2 (Delay):   CalibratedRF, CalibratedGB, CalibratedLR, CalibratedSVM, CalibratedExtraTrees, CalibratedAdaBoost → best ROC-AUC
  Agent 3 (Carrier): CalibratedGB, CalibratedRF, CalibratedExtraTrees, CalibratedLR, CalibratedDecisionTree, CalibratedAdaBoost → best ROC-AUC
All results logged to ml_models table. Best model saved to Google Drive.
"""
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, RandomForestClassifier,
                               GradientBoostingClassifier, ExtraTreesClassifier,
                               AdaBoostRegressor, AdaBoostClassifier)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db


# ── Kaggle helper ─────────────────────────────────────────────────────────────
def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    target = os.path.join(dest, filename)
    if os.path.exists(target):
        print(f"  📂 Cache hit: {filename}")
        try: return pd.read_csv(target, encoding="latin-1", on_bad_lines="skip")
        except Exception: pass
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print(f"  ℹ️  No Kaggle creds — synthetic fallback"); return None
    try:
        os.environ.update({"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY})
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        print(f"  ⬇️  Downloading {slug} …")
        api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
        if os.path.exists(target):
            df = pd.read_csv(target, encoding="latin-1", on_bad_lines="skip")
            print(f"  ✅ Loaded {len(df)} rows"); return df
        csvs = [f for f in os.listdir(dest) if f.endswith(".csv")]
        if csvs:
            df = pd.read_csv(os.path.join(dest, csvs[0]), encoding="latin-1", on_bad_lines="skip")
            print(f"  ✅ Loaded {csvs[0]}: {len(df)} rows"); return df
    except Exception as e:
        print(f"  ⚠️  Kaggle failed ({e}) — synthetic fallback")
    return None


def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    """Train all regressors, log each, save & return best by R²."""
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        p    = model.predict(X_te)
        r2   = float(r2_score(y_te, p))
        rmse = float(np.sqrt(mean_squared_error(y_te, p)))
        print(f"    {name:40s} R²={r2:.4f}  RMSE={rmse:,.0f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr)+len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model
    print(f"  🏆 Best: {best_name} (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    """Train all classifiers, log each, save & return best by ROC-AUC."""
    print(f"\n  🔬 {agent_name} — Algorithm Comparison:")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        model = CalibratedClassifierCV(base, cv=2, method="sigmoid")
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        auc   = float(roc_auc_score(y_te, proba))
        acc   = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:40s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr)+len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model = auc, name, model
    print(f"  🏆 Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc


def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # ── Agent 1: Pricing & Freight Cost (2 Kaggle Datasets: SCMS Delivery + DataCo Supply Chain) ──
    df1a = kaggle_download("apoorvwatsky/supply-chain-shipment-pricing-data",
                           "SCMS_Delivery_History_Dataset.csv")
    df1b_k = kaggle_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis",
                             "DataCoSupplyChainDataset.csv")
    if df1a is not None and "Weight (Kilograms)" in df1a.columns:
        df1a = df1a[["Weight (Kilograms)","Freight Cost (USD)","Shipment Mode"]].copy()
        df1a.columns = ["weight","base_cost","mode"]
        df1a["weight"] = pd.to_numeric(df1a["weight"].astype(str).str.replace(",", ""), errors="coerce")
        df1a["base_cost"] = pd.to_numeric(df1a["base_cost"].astype(str).str.replace(",", ""), errors="coerce")
        df1a = df1a.dropna(subset=["weight","base_cost"]).head(n)
        if len(df1a) < 50:
            df1a = None
        else:
            df1a["mode"] = df1a["mode"].map({"Air":0,"Ocean":1,"Truck":2}).fillna(1)

    if df1a is None or "weight" not in df1a.columns:
        df1a = pd.DataFrame({"weight":rng.uniform(10,450,n),
                              "base_cost":rng.uniform(2000,35000,n),
                              "mode":rng.choice([0,1,2],n,p=[0.25,0.60,0.15])})
    n1 = min(len(df1a), n)
    df1b = pd.DataFrame({"distance":rng.uniform(800,12000,n1),
                          "fuel":rng.uniform(0.90,1.38,n1),
                          "congestion":rng.choice([0,1,2],n1,p=[0.45,0.35,0.20])})
    a1 = pd.DataFrame({
        "distance":    df1b["distance"],
        "weight":      df1a["weight"].astype(float).values[:n1],
        "congestion":  df1b["congestion"],
        "fuel":        df1b["fuel"],
        "cargo_type":  rng.choice([0,1,2,3], n1),
        "port_dwell":  rng.uniform(0.5,8.0,n1),
        "target":     (df1b["distance"]*1.85 + df1a["weight"].astype(float).values[:n1]*50 +
                       df1b["congestion"]*1800)*df1b["fuel"] + rng.normal(0,400,n1),
    })

    # ── Agent 2: Delay Risk Classification (2 Kaggle Datasets: Supply Chain Analysis + Trade Logistics) ──
    raw_d1 = kaggle_download("harshsingh2209/supply-chain-analysis", "supply_chain_data.csv")
    raw_d2 = kaggle_download("victorchen/international-trade-logistics-dataset", "trade_logistics.csv")
    n2 = n
    if raw_d1 is not None and "Lead time" in raw_d1.columns:
        dwell_vals = raw_d1["Lead time"].dropna().astype(float).values
        if len(dwell_vals) < n2:
            dwell_vals = np.pad(dwell_vals, (0, n2 - len(dwell_vals)), mode="wrap")
        dwell_vals = dwell_vals[:n2]
    else:
        dwell_vals = rng.uniform(1, 9.5, n2)

    df2a = pd.DataFrame({"dwell": dwell_vals, "berth": rng.integers(5,45,n2),
                          "route_length": rng.uniform(800,12000,n2)})
    df2b = pd.DataFrame({"weather": rng.uniform(0,1,n2), "canal": rng.choice([0,1],n2,p=[0.75,0.25]),
                          "season_risk": rng.uniform(0,1,n2)})
    risk = df2a["dwell"]/9.5*0.4 + df2b["weather"]*0.35 + df2b["canal"]*0.15 + df2b["season_risk"]*0.10
    a2 = pd.DataFrame({"dwell":df2a["dwell"],"berth":df2a["berth"],
                        "route_length":df2a["route_length"],
                        "weather":df2b["weather"],"canal":df2b["canal"],
                        "season_risk":df2b["season_risk"],"delay_class":(risk>0.52).astype(int)})

    # ── Agent 3: Carrier Compliance (2 Kaggle Datasets: Carrier Perf + Shipment Audit Data) ──
    raw_c1 = kaggle_download("davidcariboo/freight-carrier-performance", "carrier_perf.csv")
    raw_c2 = kaggle_download("suraj520/logistics-shipment-audit-data", "audit_data.csv")
    n3 = n
    if raw_c1 is not None and "punctuality" in raw_c1.columns:
        punct_vals = raw_c1["punctuality"].dropna().astype(float).values
        if len(punct_vals) < n3:
            punct_vals = np.pad(punct_vals, (0, n3 - len(punct_vals)), mode="wrap")
        punct_vals = punct_vals[:n3]
    else:
        punct_vals = rng.uniform(0.70, 0.99, n3)

    df3a = pd.DataFrame({"punct": punct_vals, "avg_delay": rng.uniform(0,5,n3),
                          "complaint_rate": rng.uniform(0,0.15,n3)})
    df3b = pd.DataFrame({"fuel_sc": rng.uniform(10,22,n3), "tariff": rng.uniform(0.70,1.00,n3),
                          "docs_complete": rng.choice([0,1],n3,p=[0.15,0.85])})
    score = df3a["punct"]*0.40 + df3b["tariff"]*0.35 + df3b["docs_complete"]*0.25 - df3a["complaint_rate"]*0.5
    a3 = pd.DataFrame({"punct":df3a["punct"],"avg_delay":df3a["avg_delay"],
                        "complaint_rate":df3a["complaint_rate"],
                        "fuel_sc":df3b["fuel_sc"],"tariff":df3b["tariff"],
                        "docs_complete":df3b["docs_complete"],"compliant":(score>0.68).astype(int)})

    # Store merged records
    print("\n  💾 Storing merged records in SQLite …")
    with get_conn() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(600, n1)):
            conn.execute(
                "INSERT INTO merged_datasets (agent_target,dataset_source,origin,destination,"
                "distance_nm,weight_tons,freight_cost_usd,shipment_mode,port_congestion,"
                "dwell_time_days,berth_capacity,weather_disruption_level,"
                "carrier_punctuality,fuel_surcharge_pct,compliance_status) VALUES "
                "(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                ("All Agents","SCMS+DataCo+SupplyChain+Logistics+CarrierPerf+AuditData",
                 "Mumbai JNPT","Rotterdam",
                 float(a1["distance"].iloc[i]),float(a1["weight"].iloc[i]),
                 float(a1["target"].iloc[i]),"Ocean",
                 ["Low","Medium","High"][int(a1["congestion"].iloc[i])%3],
                 float(a2["dwell"].iloc[i]),int(a2["berth"].iloc[i]),
                 float(a2["weather"].iloc[i]),float(a3["punct"].iloc[i]),
                 float(a3["fuel_sc"].iloc[i]),
                 "Compliant" if a3["compliant"].iloc[i] else "Flagged"))
        conn.commit()
    print("  ✅ 600 merged records stored.\n")
    return a1, a2, a3


def train_all_agents():
    print("=" * 60)
    print("  🚀 FreightQuote AI — Multi-Algorithm Training Pipeline")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    # ── Agent 1: Freight Cost Regression ─────────────────────────────────────
    X1 = a1[["distance","weight","congestion","fuel","cargo_type","port_dwell"]]
    y1 = a1["target"]
    X1tr, X1te, y1tr, y1te = train_test_split(X1, y1, test_size=0.2, random_state=42)
    regressors_1 = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge":                     Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=10,random_state=42),
        "AdaBoostRegressor":         AdaBoostRegressor(n_estimators=60,random_state=42),
    }
    m1, bn1, r2_1 = compare_regressors(regressors_1, X1tr, X1te, y1tr, y1te,
                                        "Agent1_Pricing", AGENT1_MODEL_PATH)
    print(f"  → R² target ≥ 0.90: {'✅ PASS' if r2_1>=0.90 else '⚠️  BELOW TARGET'}")

    # ── Agent 2: Delay Risk Classification ───────────────────────────────────
    X2 = a2[["dwell","berth","route_length","weather","canal","season_risk"]]
    y2 = a2["delay_class"]
    X2tr, X2te, y2tr, y2te = train_test_split(X2, y2, test_size=0.2, random_state=42)
    classifiers_2 = {
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "SVC_RBF":                    Pipeline([("scl",StandardScaler()),("mdl",SVC(kernel="rbf",probability=True,random_state=42))]),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "AdaBoostClassifier":         AdaBoostClassifier(n_estimators=60,random_state=42),
    }
    m2, bn2, auc2 = compare_classifiers(classifiers_2, X2tr, X2te, y2tr, y2te,
                                         "Agent2_DelayRisk", AGENT2_MODEL_PATH)

    # ── Agent 3: Carrier Compliance Classification ────────────────────────────
    X3 = a3[["punct","avg_delay","complaint_rate","fuel_sc","tariff","docs_complete"]]
    y3 = a3["compliant"]
    X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.2, random_state=42)
    classifiers_3 = {
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=60,random_state=42,n_jobs=-1),
        "LogisticRegression":         Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "DecisionTreeClassifier":     DecisionTreeClassifier(max_depth=8,random_state=42),
        "AdaBoostClassifier":         AdaBoostClassifier(n_estimators=60,random_state=42),
    }
    m3, bn3, auc3 = compare_classifiers(classifiers_3, X3tr, X3te, y3tr, y3te,
                                         "Agent3_CarrierCompliance", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  🎉 Training Complete — Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}): R²  = {r2_1:.4f}")
    print(f"  Agent 2 ({bn2}): AUC = {auc2:.4f}")
    print(f"  Agent 3 ({bn3}): AUC = {auc3:.4f}")
    print(f"  Models saved to: {MODELS_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    train_all_agents()


Writing train_ml.py


## Step 6b — Write Main Application (`app.py`)


In [22]:
%%writefile app.py
"""
app.py — FreightQuote AI v4 FINAL (Modular Fast Engine)
Lean orchestrator — all heavy tab logic lives in agent2_freight.py, agent3_freight.py, admin_dash.py
"""
import os, json, joblib, subprocess, numpy as np, pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH
from ui_theme import apply_theme, render_header, render_card, COLORS
from auth import render_auth_portal
from db import get_conn, load_chat_history, save_chat_message
from weather_context import get_weather_report
from notifications import send_alert, get_recent_alerts
from llm_engine import (orchestrate_3_agents_query, generate_debate_and_synthesis,
                        warmup_llm, is_llm_loaded, start_background_warmup)
from agent2_freight import render_agent2_freight
from agent3_freight import render_agent3_freight
from admin_dash import render_admin_dashboard

import os as _os
_os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as _f:
    _f.write('[theme]\nbase="light"\nprimaryColor="#c9a24b"\nbackgroundColor="#f4f5f7"\nsecondaryBackgroundColor="#ffffff"\ntextColor="#1b2436"\n')

st.set_page_config(page_title="Infosys Freight Quote Portal", page_icon="🏛️", layout="wide",
                   initial_sidebar_state="expanded")
apply_theme()
start_background_warmup()

if not st.session_state.get("token"):
    render_auth_portal(); st.stop()

username  = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Logistics Manager")
is_admin  = user_role.lower() == "admin"

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(f'<div style="text-align:center;padding:10px 0;font-weight:700;font-size:18px;'
                f'color:{COLORS["text_heading"]};">🏛️ Infosys Freight Quote Portal</div>', unsafe_allow_html=True)
    st.markdown(f'<div style="text-align:center;font-size:13px;color:{COLORS["text_muted"]};'
                f'margin-bottom:12px;">User: <b>{username}</b><br>'
                f'<span style="color:#0066cc;font-weight:600;">[{user_role}]</span></div>',
                unsafe_allow_html=True)
    tabs = ["🤖 AI Copilot", "💰 Agent 1: Pricing", "🚢 Agent 2: Route/Weather",
            "✅ Agent 3: Carrier Audit", "📊 Analytics & Retrain"]
    icons = ["chat-dots-fill", "currency-dollar", "compass", "clipboard-check", "bar-chart-fill"]
    if is_admin:
        tabs.append("🛡️ Admin Dashboard"); icons.append("shield-lock-fill")
    tabs.append("🚪 Sign Out"); icons.append("box-arrow-right")
    selected_tab = option_menu(menu_title=None, options=tabs, icons=icons, default_index=0,
        styles={
            "container": {"padding": "0!important", "background-color": "transparent"},
            "nav-link": {"font-size": "13px", "text-align": "left", "margin": "3px 0",
                         "border-radius": "10px", "color": COLORS["text_main"], "font-weight": "600"},
            "nav-link-selected": {"background-color": COLORS["accent"], "color": COLORS["accent_text"],
                                  "border": f"2px solid {COLORS['border']}"},
        })

if selected_tab == "🚪 Sign Out":
    st.session_state["token"] = None; st.rerun()

render_header("Infosys Freight Quote Portal", f"Module: {selected_tab}")

# ── GPU Banner ────────────────────────────────────────────────────────────────
b1, b2 = st.columns([4, 1.2])
with b1:
    if is_llm_loaded():
        st.markdown('<div style="background:#d1fae5;border:2px solid #34d399;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:#065f46;font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Active on Tesla T4 (Qwen-2.5-3B Ready)</div>',
                    unsafe_allow_html=True)
    else:
        st.markdown('<div style="background:#bae8e8;border:2px solid #272343;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:{COLORS["text_heading"]};font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Standby — warm up before use</div>',
                    unsafe_allow_html=True)
with b2:
    if not is_llm_loaded():
        if st.button("⚡ Warm Up LLM", key="warmup_btn", use_container_width=True):
            with st.spinner("Loading Qwen-2.5-3B from Drive cache..."):
                warmup_llm()
            st.rerun()


@st.cache_resource
def load_agents():
    if not os.path.exists(AGENT1_MODEL_PATH) or not os.path.exists(AGENT2_MODEL_PATH) or not os.path.exists(AGENT3_MODEL_PATH):
        try:
            from train_ml import train_all_agents
            train_all_agents()
        except Exception as e:
            print(f"Auto-training note: {e}")
    m1 = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2 = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m3 = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2, m3

agent1_m, agent2_m, agent3_m = load_agents()


def confidence_band(model, X_row):
    if model is None:
        return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba([X_row])[0][1])
    else:
        prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob + z**2/(2*n) - z*((prob*(1-prob)+z**2/(4*n))/n)**0.5) / (1+z**2/n))
    hi = min(1.0, (prob + z**2/(2*n) + z*((prob*(1-prob)+z**2/(4*n))/n)**0.5) / (1+z**2/n))
    return prob, lo, hi


# Shared context (built once per page load)
with get_conn() as conn:
    n_quotes   = conn.execute("SELECT count(*) FROM quotes").fetchone()[0]
    n_ships    = conn.execute("SELECT count(*) FROM shipments").fetchone()[0]
    n_carriers = conn.execute("SELECT count(*) FROM carriers").fetchone()[0]
    n_alerts   = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]

db_stats = {"quotes": n_quotes, "shipments": n_ships,
            "carriers": n_carriers, "alerts": n_alerts}
a1_ctx = {"base_rate_usd": 18500, "congestion": "High", "fuel_surcharge_pct": 13.5}
a2_ctx = {"dwell_days": 3.8, "canal_queue": True, "delay_risk_pct": 68}
a3_ctx = {"carrier": "Maersk", "punctuality": 0.94, "compliance": "Passed"}

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AI COPILOT
# ─────────────────────────────────────────────────────────────────────────────
if selected_tab == "🤖 AI Copilot":
    render_card('<h3 style="margin:0 0 6px;">💬 Unified AI Copilot — Total Logistics Intelligence</h3>'
                '<p style="margin:0;color:#64748b;font-size:13px;">Powered by Qwen-2.5-3B on T4. '
                'All answers use live DB stats, port weather, ML scores & carrier data.</p>')

    if "copilot_history" not in st.session_state:
        hist = load_chat_history(username, get_conn)
        if not hist:
            msg = "Welcome to FreightQuote AI Copilot! Ask about pricing, routes, carriers, or delays."
            save_chat_message(username, "assistant", msg, get_conn)
            hist = [{"role": "assistant", "content": msg}]
        st.session_state["copilot_history"] = hist

    for m in st.session_state["copilot_history"]:
        bg = "#e3f6f5" if m["role"] == "user" else "white"
        label = "🧑 You" if m["role"] == "user" else "⚡ Copilot"
        st.markdown(f'<div class="pn-card" style="background:{bg};border-left:5px solid '
                    f'{COLORS["accent"] if m["role"]=="user" else COLORS["border"]};">'
                    f'<b>{label}:</b><br>{m["content"]}</div>', unsafe_allow_html=True)

    inp_col, clr_col = st.columns([8, 1])
    with inp_col:
        with st.form("copilot_form", clear_on_submit=True):
            user_q  = st.text_input("", placeholder="e.g. 'Why is Shanghai→Rotterdam costly right now?'")
            fa, fb  = st.columns([3, 1])
            with fa: submit = st.form_submit_button("🚀 Ask Copilot")
            with fb: debate = st.form_submit_button("🔍 Debate View")
    with clr_col:
        if st.button("🗑️", help="Clear history"):
            from db import clear_chat_history
            clear_chat_history(username, get_conn)
            st.session_state["copilot_history"] = []; st.rerun()

    if (submit or debate) and user_q.strip():
        save_chat_message(username, "user", user_q, get_conn)
        st.session_state["copilot_history"].append({"role": "user", "content": user_q})
        llm_ready = is_llm_loaded()
        spinner_msg = ("⚡ Generating with Qwen2.5-3B..." if llm_ready
                       else "🔄 LLM warming up — answering instantly with rule-based analysis...")
        if debate:
            with st.spinner(spinner_msg):
                res = generate_debate_and_synthesis(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
            dc1, dc2, dc3 = st.columns(3)
            for col, key, label, color in [
                (dc1, "agent1", "Pricing & Congestion", COLORS["accent"]),
                (dc2, "agent2", "Route & Weather", "#34d399"),
                (dc3, "agent3", "Carrier Audit", "#f87171"),
            ]:
                col.markdown(f'<div class="pn-card" style="border-top:4px solid {color};">'
                             f'<span class="agent-badge">{label}</span><br><br>{res[key]}</div>',
                             unsafe_allow_html=True)
            ans = f"**Executive Synthesis:** {res['synthesis']}"
        else:
            with st.spinner(spinner_msg):
                ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
        if not llm_ready:
            st.caption("⏳ Real Qwen2.5-3B response will be used automatically once GPU warmup finishes — just ask again.")
        save_chat_message(username, "assistant", ans, get_conn)
        st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
        st.rerun()

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 1 — PRICING
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "💰 Agent 1: Pricing":
    render_card('<h3 style="margin:0;">💰 Agent 1: Global Freight Pricing & Port Congestion</h3>')
    c1, c2 = st.columns(2)
    with c1:
        dist   = st.number_input("Distance (nm)", 500.0, 20000.0, 10500.0)
        weight = st.number_input("Cargo Weight (tons)", 1.0, 500.0, 45.0)
        cong   = st.selectbox("Congestion Level", ["Low (0)", "Medium (1)", "High (2)"], index=2)
        fuel   = st.slider("Fuel Index", 0.9, 1.6, 1.18)
        cargo  = st.selectbox("Cargo Type", ["General (0)", "Perishable (1)", "Hazmat (2)", "Heavy (3)"])
        dwell  = st.number_input("Port Dwell (days)", 0.5, 14.0, 3.8)
        cong_v  = int(cong.split("(")[1].replace(")", ""))
        cargo_v = int(cargo.split("(")[1].replace(")", ""))
    with c2:
        if st.button("⚡ Generate Quote"):
            row = [dist, weight, cong_v, fuel, cargo_v, dwell]
            if agent1_m:
                preds = [t.predict([row])[0] for t in agent1_m.estimators_]
                mean_p, std_p = float(np.mean(preds)), float(np.std(preds))
            else:
                mean_p = dist * 1.8 + weight * 48 + cong_v * 1600; std_p = mean_p * 0.05
            lo95, hi95 = mean_p - 1.96*std_p, mean_p + 1.96*std_p
            st.markdown(
                f'<div style="background:{COLORS["accent"]};padding:16px;border-radius:12px;'
                f'border:2px solid {COLORS["border"]};">'
                f'<span class="agent-badge">Agent 1 Estimate</span>'
                f'<h2 style="color:{COLORS["text_heading"]};margin:8px 0 0;">${mean_p:,.0f}</h2>'
                f'<p style="font-weight:600;margin:4px 0;">95% CI: ${lo95:,.0f} — ${hi95:,.0f}</p>'
                f'<p style="margin:0;font-size:12px;">±{std_p/mean_p*100:.1f}% uncertainty</p>'
                f'</div>', unsafe_allow_html=True)
            send_alert("In-App", username, "Quote Generated", f"${mean_p:,.0f}")

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 2 — ROUTE/WEATHER (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🚢 Agent 2: Route/Weather":
    render_agent2_freight(agent2_m, username, db_stats, a1_ctx, a3_ctx,
                          send_alert, get_conn, confidence_band)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 3 — CARRIER AUDIT (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "✅ Agent 3: Carrier Audit":
    render_agent3_freight(agent3_m, username, confidence_band)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ANALYTICS & RETRAIN
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "📊 Analytics & Retrain":
    render_card('<h3 style="margin:0;">📊 Enterprise Analytics & Model Management</h3>')
    kc = st.columns(4)
    for col, icon, label, val in [
        (kc[0], "📋", "Total Quotes",   n_quotes),
        (kc[1], "🚢", "Shipments",      n_ships),
        (kc[2], "✅", "Carriers",       n_carriers),
        (kc[3], "🔔", "Alerts Sent",    n_alerts),
    ]:
        col.markdown(f'<div class="pn-card" style="text-align:center;padding:14px;">'
                     f'<div style="font-size:26px;">{icon}</div>'
                     f'<h2 style="margin:4px 0;">{val}</h2>'
                     f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
                     f'</div>', unsafe_allow_html=True)
    st.markdown("---")
    mc1, mc2 = st.columns([1, 1.5])
    with mc1:
        render_card('<h4 style="margin:0 0 8px;">🔄 1-Click Retrain</h4>')
        if st.button("🔄 Retrain All Agents Now"):
            with st.spinner("Training... (~2-3 min)"):
                res = subprocess.run(["python", "train_ml.py"], capture_output=True, text=True, timeout=300)
            load_agents.clear()
            (st.success if res.returncode == 0 else st.error)(
                "✅ All agents retrained!" if res.returncode == 0 else "❌ Training failed.")
            st.code((res.stdout if res.returncode == 0 else res.stderr)[-1000:])
    with mc2:
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql("SELECT agent_name,model_name,r2_score,accuracy,"
                                    "training_rows,created_at FROM ml_models ORDER BY id DESC", conn)
                st.dataframe(ml_df, use_container_width=True, hide_index=True)
            except Exception:
                st.info("No model history yet.")
    st.markdown("---")
    render_card('<h4 style="margin:0 0 8px;">🔔 Recent Alerts</h4>')
    for a in get_recent_alerts(10):
        st.markdown(f'<div style="border-bottom:1px solid #bae8e8;padding:5px 0;font-size:13px;">'
                    f'<b>[{a[1].upper()}]</b> {a[3]} '
                    f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
                    unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ADMIN DASHBOARD (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🛡️ Admin Dashboard":
    if not is_admin:
        st.error("🔒 Admin access required.")
    else:
        render_admin_dashboard(project="freight")


Overwriting app.py


## Step 7 — Launch Streamlit App via ngrok


In [25]:
import subprocess, time, os
from pyngrok import ngrok
try:
    from config import NGROK_AUTHTOKEN as NGROK_AUTH_TOKEN
except ImportError:
    from config import NGROK_AUTH_TOKEN

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print("🚀 App Published at:", public_url)
else:
    print("Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501", "--server.headless=true"])
print("✅ Streamlit started (PID:", process.pid, ")")


🚀 App Published at: https://snitch-disperser-sterling.ngrok-free.dev
✅ Streamlit started (PID: 3830 )


## Step 8 — Stop Application & Free GPU Memory


In [24]:
try:
    process.terminate()
    ngrok.kill()
    print("🛑 Streamlit and ngrok terminated successfully.")
except Exception as e:
    print("Info:", e)


🛑 Streamlit and ngrok terminated successfully.
